# `metrics.py` — Evaluation Metrics for Multi-Aspect Sentiment Analysis

## Purpose
Provides three evaluation classes used throughout training, ablation studies, and the final
evaluation pipeline.

| Class | Role |
|-------|------|
| `AspectSentimentEvaluator` | Per-aspect Accuracy, Macro-F1, Weighted-F1, MCC, confusion matrices, LaTeX output |
| `ErrorAnalyzer` | Finds misclassified samples and breaks them down by aspect and error type |
| `MixedSentimentEvaluator` | Identifies reviews with conflicting sentiments across aspects; measures how well the model resolves them |

## Why Macro-F1 as the Primary Metric?
The dataset is class-imbalanced (positive >> neutral > negative for most aspects).
- **Macro-F1** treats every class equally — poor performance on the minority `negative` class
  is just as costly as poor performance on the majority `positive` class.
- **Weighted-F1** can be deceptively high when the model simply predicts the majority class,
  so it is reported as a secondary metric only.
- **MCC** (Matthews Correlation Coefficient) provides a single balanced score even with
  very different class sizes. Range: -1 (worst) to +1 (perfect); 0 = random.

In [ ]:
# ── Evaluation metrics from scikit-learn ────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    matthews_corrcoef,
    classification_report,
)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

print('[OK] Imports successful')

## Class 1 — `AspectSentimentEvaluator`

Stores per-aspect metrics in a dict and provides methods to:
- **`evaluate_aspect(y_true, y_pred, name)`** — compute all metrics and store them
- **`print_results(name)`** — formatted console output with per-class breakdown and text confusion matrix
- **`plot_confusion_matrix(name)`** — seaborn heatmap for one aspect
- **`plot_all_confusion_matrices()`** — grid of heatmaps for all aspects
- **`compare_aspects()`** — cross-aspect summary table and bar charts
- **`generate_latex_table()`** — LaTeX `tabular` block for the thesis
- **`save_results(path)`** — JSON serialisation (numpy arrays converted to lists)

### Label encoding used throughout: `0 = negative`, `1 = neutral`, `2 = positive`

In [ ]:
class AspectSentimentEvaluator:
    """
    Comprehensive evaluation for imbalanced multi-aspect sentiment.

    Usage
    -----
    evaluator = AspectSentimentEvaluator(['smell', 'texture', 'price'])
    evaluator.evaluate_aspect(y_true, y_pred, 'smell')
    evaluator.print_results('smell')
    print(evaluator.generate_latex_table())
    """

    def __init__(self, aspect_names: list):
        self.aspect_names = aspect_names
        self.results      = {}   # filled by evaluate_aspect(); keyed by aspect name

    # ── Core Evaluation ──────────────────────────────────────────────────────

    def evaluate_aspect(self, y_true, y_pred, aspect_name: str) -> dict:
        """
        Compute all metrics for a single aspect and cache them.

        Parameters
        ----------
        y_true      : array-like of int  — ground-truth labels
        y_pred      : array-like of int  — model predictions
        aspect_name : str                — key used to store/retrieve results

        Returns
        -------
        dict containing all computed metrics
        """
        # Overall fraction of samples classified correctly
        accuracy = accuracy_score(y_true, y_pred)

        # Per-class metrics: one value per class [negative, neutral, positive]
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true, y_pred, average=None, zero_division=0, labels=[0, 1, 2]
        )

        # Macro-F1: unweighted mean across classes — PRIMARY metric.
        # Poor perf on rare negative class costs just as much as poor perf on majority positive.
        macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average='macro', zero_division=0, labels=[0, 1, 2]
        )

        # Weighted-F1: support-weighted mean — secondary metric.
        # Can appear high when model is good at only the majority class.
        w_p, w_r, weighted_f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0, labels=[0, 1, 2]
        )

        # MCC handles any class-size distribution; +1=perfect, 0=random, -1=inverse
        mcc = matthews_corrcoef(y_true, y_pred)

        cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

        self.results[aspect_name] = {
            'accuracy'           : float(accuracy),
            'macro_precision'    : float(macro_p),
            'macro_recall'       : float(macro_r),
            'macro_f1'           : float(macro_f1),
            'weighted_precision' : float(w_p),
            'weighted_recall'    : float(w_r),
            'weighted_f1'        : float(weighted_f1),
            'mcc'                : float(mcc),
            'per_class_precision': precision,
            'per_class_recall'   : recall,
            'per_class_f1'       : f1,
            'support'            : support,
            'confusion_matrix'   : cm,
        }
        return self.results[aspect_name]

    # ── Console Output ────────────────────────────────────────────────────────

    def print_results(self, aspect_name: str):
        """Pretty-print all metrics for one aspect, including a text confusion matrix."""
        if aspect_name not in self.results:
            print(f'No results for {aspect_name}')
            return

        r = self.results[aspect_name]
        print(f"\n{'='*70}")
        print(f"Results for {aspect_name.upper()}")
        print(f"{'='*70}")
        print(f"Accuracy    : {r['accuracy']:.4f}")
        print(f"Macro F1    : {r['macro_f1']:.4f}   <- primary metric")
        print(f"Weighted F1 : {r['weighted_f1']:.4f}")
        print(f"MCC         : {r['mcc']:.4f}")

        print(f"\n{'Class':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Support'}")
        print(f"{'-'*70}")
        for i, cls in enumerate(['Negative', 'Neutral', 'Positive']):
            print(f"{cls:<12} {r['per_class_precision'][i]:<12.4f} "
                  f"{r['per_class_recall'][i]:<12.4f} "
                  f"{r['per_class_f1'][i]:<12.4f} {int(r['support'][i])}")

        cm = r['confusion_matrix']
        print("\nConfusion Matrix (rows=True label, cols=Predicted label):")
        print("              Pred Neg  Pred Neu  Pred Pos")
        for row_lbl, row in zip(['True Neg', 'True Neu', 'True Pos'], cm):
            print(f"{row_lbl}      {'  '.join(f'{v:8d}' for v in row)}")

    # ── Plotting ──────────────────────────────────────────────────────────────

    def plot_confusion_matrix(self, aspect_name: str, save_path=None):
        """Seaborn heatmap for one aspect's confusion matrix."""
        if aspect_name not in self.results:
            print(f'No results for {aspect_name}')
            return
        cm = self.results[aspect_name]['confusion_matrix']
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Negative', 'Neutral', 'Positive'],
                    yticklabels=['Negative', 'Neutral', 'Positive'])
        plt.title(f'Confusion Matrix — {aspect_name}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f'Saved to {save_path}')
        else:
            plt.show()
        plt.close()

    def plot_all_confusion_matrices(self, save_dir=None):
        """Plot all stored aspects in a grid of confusion matrix heatmaps."""
        if not self.results:
            print('No results to plot')
            return
        n = len(self.results)
        nc = min(3, n)
        nr = (n + nc - 1) // nc
        fig, axes = plt.subplots(nr, nc, figsize=(5 * nc, 5 * nr))
        axes_flat = np.array(axes).flatten() if n > 1 else [axes]
        for idx, (asp, r) in enumerate(self.results.items()):
            sns.heatmap(r['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                        xticklabels=['Neg', 'Neu', 'Pos'],
                        yticklabels=['Neg', 'Neu', 'Pos'],
                        ax=axes_flat[idx], cbar=False)
            axes_flat[idx].set_title(f"{asp}\nF1={r['macro_f1']:.3f}")
        for idx in range(n, len(axes_flat)):
            axes_flat[idx].axis('off')
        plt.tight_layout()
        if save_dir:
            p = Path(save_dir) / 'all_confusion_matrices.png'
            plt.savefig(p, dpi=300, bbox_inches='tight')
            print(f'Saved to {p}')
        else:
            plt.show()
        plt.close()

    def compare_aspects(self):
        """Print cross-aspect comparison table and show bar charts."""
        if not self.results:
            print('No results')
            return
        print(f"\n{'='*90}")
        print('Performance Comparison Across Aspects')
        print(f"{'='*90}")
        print(f"{'Aspect':<15} {'Accuracy':<12} {'Macro-F1':<12} {'Weighted-F1':<12} {'MCC'}")
        print(f"{'-'*90}")
        for asp in sorted(self.results):
            r = self.results[asp]
            print(f"{asp:<15} {r['accuracy']:<12.4f} "
                  f"{r['macro_f1']:<12.4f} {r['weighted_f1']:<12.4f} {r['mcc']:.4f}")
        aspects  = list(self.results.keys())
        metrics_ = ['accuracy', 'macro_f1', 'weighted_f1', 'mcc']
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        for idx, metric in enumerate(metrics_):
            ax     = axes.flatten()[idx]
            values = [self.results[asp][metric] for asp in aspects]
            ax.bar(aspects, values, color='steelblue', alpha=0.75)
            ax.set_title(metric.replace('_', ' ').title())
            ax.set_ylabel('Score')
            ax.set_ylim(0, 1.05)
            ax.tick_params(axis='x', rotation=45)
            for i, v in enumerate(values):
                ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)
        plt.tight_layout()
        plt.show()

    # ── LaTeX / Serialisation ─────────────────────────────────────────────────

    def generate_latex_table(self) -> str:
        """Return a LaTeX tabular block ready to paste into a .tex file."""
        if not self.results:
            return ''
        lines = [
            r'\begin{table}[h]', r'\centering',
            r'\begin{tabular}{lcccc}', r'\hline',
            r'Aspect & Accuracy & Macro F1 & Weighted F1 & MCC \\', r'\hline',
        ]
        for asp in sorted(self.results):
            r = self.results[asp]
            lines.append(
                f"{asp} & {r['accuracy']:.4f} & {r['macro_f1']:.4f} & "
                f"{r['weighted_f1']:.4f} & {r['mcc']:.4f} \\\\"
            )
        lines += [
            r'\hline', r'\end{tabular}',
            r'\caption{Multi-aspect sentiment analysis results}',
            r'\label{tab:results}', r'\end{table}',
        ]
        return '\n'.join(lines)

    def save_results(self, save_path: str):
        """Serialise all metrics to JSON (numpy arrays are converted to lists)."""
        import json
        serialisable = {}
        for asp, mets in self.results.items():
            serialisable[asp] = {
                k: v.tolist() if isinstance(v, np.ndarray)
                   else float(v) if isinstance(v, (np.float32, np.float64))
                   else v
                for k, v in mets.items()
            }
        with open(save_path, 'w') as f:
            json.dump(serialisable, f, indent=2)
        print(f'Results saved to {save_path}')

## Class 2 — `ErrorAnalyzer`

After training, understanding **which** samples were misclassified guides model improvement.
`ErrorAnalyzer.analyze_errors()` takes the raw texts alongside true/pred labels and produces:
1. **Error rate per aspect** — which aspects are hardest to classify
2. **Error-type breakdown** — e.g. `negative->positive` (most dangerous mistake: completely wrong polarity)
   vs. `positive->neutral` (less severe: close to correct)
3. **Optional CSV export** for manual inspection of specific misclassified examples

In [ ]:
class ErrorAnalyzer:
    """
    Analyse prediction errors to surface actionable improvement insights.

    Usage
    -----
    analyzer = ErrorAnalyzer(aspect_names, class_names=['negative','neutral','positive'])
    errors   = analyzer.analyze_errors(texts, y_true, y_pred, aspects, save_path='errors.csv')
    """

    def __init__(self, aspect_names: list, class_names: list):
        self.aspect_names = aspect_names
        # Ordered list matching label integers: class_names[0]='negative', etc.
        self.class_names  = class_names

    def analyze_errors(
        self,
        texts    : list,
        y_true   : list,
        y_pred   : list,
        aspects  : list,
        save_path: str = None,
    ) -> list:
        """
        Find all misclassified samples and report error statistics.

        Parameters
        ----------
        texts     : list[str]  — review text for each sample
        y_true    : list[int]  — ground-truth label for each sample
        y_pred    : list[int]  — model prediction for each sample
        aspects   : list[str]  — aspect name for each sample
        save_path : str | None — if given, write CSV of error records here

        Returns
        -------
        errors : list of dicts, one per misclassified sample
        """
        # ── Collect all misclassified samples ─────────────────────────────────
        errors = [
            {
                'text'      : texts[i],
                'aspect'    : aspects[i],
                'true_label': self.class_names[y_true[i]],
                'pred_label': self.class_names[y_pred[i]],
                # 'true->pred' format makes error-type counting easy to read
                'error_type': f"{self.class_names[y_true[i]]}->{self.class_names[y_pred[i]]}",
            }
            for i in range(len(texts))
            if y_true[i] != y_pred[i]
        ]

        total = len(texts)
        print(f'\nTotal errors: {len(errors)} / {total}  ({len(errors)/total*100:.2f}%)')

        # ── Error rate per aspect ─────────────────────────────────────────────
        print('\nError rate by aspect:')
        aspect_errors = Counter(e['aspect'] for e in errors)
        for asp, cnt in sorted(aspect_errors.items()):
            total_asp = sum(1 for a in aspects if a == asp)
            print(f'  {asp}: {cnt}/{total_asp}  ({cnt/total_asp*100:.1f}%)')

        # ── Error-type distribution ───────────────────────────────────────────
        # 'negative->positive' is the worst: completely wrong polarity
        print('\nError type distribution (true->pred):')
        for etype, cnt in sorted(
            Counter(e['error_type'] for e in errors).items(),
            key=lambda x: -x[1]
        ):
            print(f'  {etype}: {cnt}  ({cnt/len(errors)*100:.1f}%)')

        # ── Optional CSV export ───────────────────────────────────────────────
        if save_path:
            import pandas as pd
            pd.DataFrame(errors).to_csv(save_path, index=False)
            print(f'\nError CSV saved to {save_path}')

        return errors

## Class 3 — `MixedSentimentEvaluator`

This is a **core research contribution** of the ClearView project.

A *mixed-sentiment review* contains conflicting sentiments across different aspects —
e.g. *"great colour but terrible smell"* → `colour=positive`, `smell=negative`.
These are the hardest cases for any sentiment model.

### Two evaluation stages

1. **`identify_mixed_sentiment_reviews(reviews_data)`**
   - Scans ground-truth labels to find reviews with conflicting sentiments.
   - Classifies each conflict as:
     - `positive_negative` — has both pos and neg, no neutral
     - `positive_neutral_negative` — all three sentiments present
     - `neutral_with_extremes` — neutral + pos or neg only

2. **`evaluate_mixed_sentiment_resolution(y_true_dict, y_pred_dict)`**
   - Measures how many mixed-sentiment reviews the model got **completely right**
     (all aspects correct = review-level accuracy).
   - Also measures aspect-level accuracy and mixed detection rate within those reviews.

This is used in **Ablation A6** to demonstrate that GCN + aspect attention improves
performance specifically on the hardest mixed-sentiment cases.

In [ ]:
class MixedSentimentEvaluator:
    """
    Evaluates the model's ability to correctly resolve conflicting sentiments
    across aspects within a single review.

    Label encoding: 0 = negative, 1 = neutral, 2 = positive
    """

    def __init__(self, aspect_names: list):
        self.aspect_names = aspect_names
        self.class_names  = ['negative', 'neutral', 'positive']

    # ── Stage 1: Identify mixed reviews ──────────────────────────────────────

    def identify_mixed_sentiment_reviews(self, reviews_data: list) -> tuple:
        """
        Scan ground-truth labels to find reviews with conflicting aspect sentiments.

        Parameters
        ----------
        reviews_data : list of dicts, each with:
            {
              'review_id': str,
              'text'     : str,
              'aspects'  : {aspect_name: label_int_or_None, ...}
            }

        Returns
        -------
        mixed_ids : list of review_id strings that are mixed-sentiment
        stats     : dict with counts and percentages
        """
        mixed_ids = []
        stats = {
            'total_reviews'         : len(reviews_data),
            'mixed_sentiment_reviews': 0,
            'single_aspect_reviews' : 0,
            'multi_aspect_reviews'  : 0,
            'conflict_types': {
                'positive_negative'        : 0,   # pos + neg only
                'positive_neutral_negative': 0,   # all three
                'neutral_with_extremes'    : 0,   # neutral + one extreme
            }
        }

        for review in reviews_data:
            # Only consider aspects that have a non-None label
            active = {asp: lbl for asp, lbl in review['aspects'].items()
                      if lbl is not None}
            if not active:
                continue
            if len(active) == 1:
                stats['single_aspect_reviews'] += 1
                continue

            stats['multi_aspect_reviews'] += 1
            sentiments = set(active.values())
            has_pos    = 2 in sentiments
            has_neu    = 1 in sentiments
            has_neg    = 0 in sentiments

            is_mixed = False
            if has_pos and has_neg:
                # Direct polarity conflict — the most interesting case
                is_mixed = True
                if has_neu:
                    stats['conflict_types']['positive_neutral_negative'] += 1
                else:
                    stats['conflict_types']['positive_negative'] += 1
            elif has_neu and (has_pos or has_neg):
                # Neutral mixed with a polarised aspect (softer conflict)
                is_mixed = True
                stats['conflict_types']['neutral_with_extremes'] += 1

            if is_mixed:
                mixed_ids.append(review['review_id'])
                stats['mixed_sentiment_reviews'] += 1

        total_multi = stats['multi_aspect_reviews'] or 1
        stats['mixed_percentage_of_multi'] = (
            stats['mixed_sentiment_reviews'] / total_multi * 100
        )
        stats['mixed_percentage_of_total'] = (
            stats['mixed_sentiment_reviews'] / (stats['total_reviews'] or 1) * 100
        )

        return mixed_ids, stats

    # ── Stage 2: Evaluate model on mixed reviews ──────────────────────────────

    def evaluate_mixed_sentiment_resolution(
        self, y_true_dict: dict, y_pred_dict: dict
    ) -> dict:
        """
        Measure how well the model handles mixed-sentiment reviews.

        Parameters
        ----------
        y_true_dict : {review_id: {aspect: true_label}}
        y_pred_dict : {review_id: {aspect: pred_label}}

        Returns
        -------
        dict with mixed_review_count, mixed_review_accuracy, mixed_aspect_accuracy, stats
        """
        reviews_data = [
            {'review_id': rid, 'text': '', 'aspects': labels}
            for rid, labels in y_true_dict.items()
        ]
        mixed_ids, mixed_stats = self.identify_mixed_sentiment_reviews(reviews_data)

        if not mixed_ids:
            print('Warning: no mixed-sentiment reviews found')
            return {
                'mixed_review_count'   : 0,
                'mixed_review_accuracy': 0.0,
                'mixed_aspect_accuracy': 0.0,
                'stats'                : mixed_stats,
            }

        correct_reviews  = 0
        total_aspects    = 0
        correct_aspects  = 0

        for rid in mixed_ids:
            if rid not in y_pred_dict:
                continue
            true_asp = y_true_dict[rid]
            pred_asp = y_pred_dict[rid]
            all_ok   = True

            for asp, true_lbl in true_asp.items():
                if asp in pred_asp:
                    total_aspects += 1
                    if true_lbl == pred_asp[asp]:
                        correct_aspects += 1
                    else:
                        all_ok = False
                else:
                    all_ok = False

            if all_ok:
                correct_reviews += 1

        return {
            'mixed_review_count'    : len(mixed_ids),
            'mixed_review_accuracy' : correct_reviews / len(mixed_ids) * 100,
            'mixed_aspect_accuracy' : correct_aspects / (total_aspects or 1) * 100,
            'correct_mixed_aspects' : correct_aspects,
            'total_mixed_aspects'   : total_aspects,
            'stats'                 : mixed_stats,
        }

    # ── Reporting ─────────────────────────────────────────────────────────────

    def print_mixed_sentiment_results(self, metrics: dict):
        """Formatted console output for the MSR evaluation results."""
        print(f"\n{'='*70}")
        print('MIXED SENTIMENT RESOLUTION EVALUATION')
        print(f"{'='*70}")
        s = metrics['stats']
        print(f"  Total reviews              : {s['total_reviews']}")
        print(f"  Multi-aspect reviews       : {s['multi_aspect_reviews']}")
        print(f"  Mixed-sentiment reviews    : {s['mixed_sentiment_reviews']}")
        print(f"  Mixed % (of multi-aspect)  : {s['mixed_percentage_of_multi']:.2f}%")
        print(f"  Mixed % (of total)         : {s['mixed_percentage_of_total']:.2f}%")
        ct = s['conflict_types']
        print(f"\n  Pos+Neg only               : {ct['positive_negative']}")
        print(f"  All three sentiments       : {ct['positive_neutral_negative']}")
        print(f"  Neutral with extremes      : {ct['neutral_with_extremes']}")
        if metrics['mixed_review_count'] > 0:
            print(f"\n  Total mixed reviews tested : {metrics['mixed_review_count']}")
            print(f"  Review-level accuracy      : {metrics['mixed_review_accuracy']:.2f}%")
            print(f"    (ALL aspects must be correct for 1 review-level correct)")
            print(f"  Aspect-level accuracy      : {metrics['mixed_aspect_accuracy']:.2f}%")
            print(f"    ({metrics['correct_mixed_aspects']}/{metrics['total_mixed_aspects']} aspects correct)")
        print(f"{'='*70}\n")

    def save_mixed_sentiment_analysis(self, metrics: dict, save_path: str):
        """Persist the MSR metrics dict to a JSON file."""
        import json
        with open(save_path, 'w') as f:
            json.dump(metrics, f, indent=2)
        print(f'MSR analysis saved to {save_path}')

## Quick Test / Verification

Uses only **synthetic data** — no trained model or dataset files required.

Checks:
- `AspectSentimentEvaluator` produces a 3x3 confusion matrix and correct metric keys
- `ErrorAnalyzer` finds exactly the 3 misclassified samples and reports correct types
- `MixedSentimentEvaluator` correctly labels reviews A and C as mixed (not B)
- MSR accuracy = 50% because only review A is fully correct (C gets `smell` wrong)

In [ ]:
import json

# ── 1. AspectSentimentEvaluator ───────────────────────────────────────────────
print('=== 1. AspectSentimentEvaluator ===')

y_true = np.array([0, 0, 1, 1, 2, 2, 2, 0, 1, 2])
y_pred = np.array([0, 1, 1, 2, 2, 2, 1, 0, 1, 2])  # 3 errors at indices 1, 3, 6

evaluator = AspectSentimentEvaluator(['smell', 'texture', 'price'])
results   = evaluator.evaluate_aspect(y_true, y_pred, 'smell')

assert 'macro_f1' in results,           'Missing macro_f1'
assert 'confusion_matrix' in results,   'Missing confusion_matrix'
assert results['confusion_matrix'].shape == (3, 3), 'CM must be 3x3'
assert 0.0 <= results['macro_f1'] <= 1.0, 'macro_f1 out of range'

evaluator.print_results('smell')
print('\nLaTeX output:')
print(evaluator.generate_latex_table())
print('\n[OK] AspectSentimentEvaluator passed')


# ── 2. ErrorAnalyzer ──────────────────────────────────────────────────────────
print('\n=== 2. ErrorAnalyzer ===')

texts   = [f'Review {i}' for i in range(10)]
aspects = ['smell'] * 5 + ['texture'] * 5

analyzer = ErrorAnalyzer(['smell', 'texture'], ['negative', 'neutral', 'positive'])
errors   = analyzer.analyze_errors(texts, y_true.tolist(), y_pred.tolist(), aspects)

assert len(errors) == 3, f'Expected 3 errors, got {len(errors)}'
assert all('error_type' in e for e in errors), 'Missing error_type in some records'
print(f'\n[OK] ErrorAnalyzer found {len(errors)} errors as expected')


# ── 3. MixedSentimentEvaluator ───────────────────────────────────────────────
print('\n=== 3. MixedSentimentEvaluator ===')

# Review A: colour=positive(2), smell=negative(0)  -> mixed (pos+neg)
# Review B: texture=positive(2), price=positive(2) -> NOT mixed (all positive)
# Review C: colour=neutral(1),   smell=positive(2) -> mixed (neutral with extreme)
reviews_data = [
    {'review_id': 'A', 'text': 'great colour but terrible smell',
     'aspects': {'colour': 2, 'smell': 0}},
    {'review_id': 'B', 'text': 'great texture and great price',
     'aspects': {'texture': 2, 'price': 2}},
    {'review_id': 'C', 'text': 'ok colour but lovely smell',
     'aspects': {'colour': 1, 'smell': 2}},
]

msr_eval = MixedSentimentEvaluator(['colour', 'smell', 'texture', 'price'])
mixed_ids, stats = msr_eval.identify_mixed_sentiment_reviews(reviews_data)

assert set(mixed_ids) == {'A', 'C'}, f'Expected {{A, C}}, got {set(mixed_ids)}'
assert stats['conflict_types']['positive_negative'] == 1      # Review A
assert stats['conflict_types']['neutral_with_extremes'] == 1  # Review C
print(f'Mixed reviews identified correctly: {mixed_ids}')

# Perfect predictor for A; makes 1 error on C (predicts smell=1 instead of 2)
y_true_d = {'A': {'colour': 2, 'smell': 0}, 'B': {'texture': 2, 'price': 2},
            'C': {'colour': 1, 'smell': 2}}
y_pred_d = {'A': {'colour': 2, 'smell': 0}, 'B': {'texture': 2, 'price': 2},
            'C': {'colour': 1, 'smell': 1}}  # C: smell wrong

msr_metrics = msr_eval.evaluate_mixed_sentiment_resolution(y_true_d, y_pred_d)
msr_eval.print_mixed_sentiment_results(msr_metrics)

assert msr_metrics['mixed_review_count'] == 2, 'Expected 2 mixed reviews'
assert msr_metrics['mixed_review_accuracy'] == 50.0, 'Only A fully correct -> 50%'

print('[OK] MixedSentimentEvaluator passed all assertions')
print('\nAll metrics.py tests PASSED.')